# M2-B2 — Mission étoile : LLM-as-a-judge pour l'anonymisation — SUJET

> **Bonus optionnel, non noté.** Ton anonymisation (spaCy + regex) rate une
> partie des **noms FR** (~12 % du corpus). Tu vas brancher un **LLM local**
> comme **filet de sécurité** qui relit les commentaires **anonymisés** pour :
> 1. repérer les **PII résiduelles** (surtout noms FR ratés par `en_core_web_md`)
> 2. scorer la **lisibilité** du texte masqué.

## 🚫 Garde-fou
> Le LLM **complète** spaCy/regex, il ne le **remplace pas** : non déterministe,
> coûteux, il hallucine. On ne déploie jamais une anonymisation 100 % LLM.
> Backend **Ollama local** → les PII ne quittent pas ta machine.

## Pré-requis
```bash
# 1) Installer Ollama : https://ollama.com  puis :
ollama pull llama3.2:3b
# 2) Décommenter la dep bonus dans requirements.txt (requests)
pip install requests
```
> Pas d'Ollama ? Tu coderas un **`MOCK_MODE`** (repli heuristique) pour que le
> notebook tourne quand même (TODO plus bas).

## 1. Détection d'Ollama (fourni)

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "llama3.2:3b"
TIMEOUT_S = 60


def _ollama_available() -> bool:
    try:
        return requests.get("http://localhost:11434/api/tags",
                            timeout=3).status_code == 200
    except requests.RequestException:
        return False


MOCK_MODE = not _ollama_available()
print("MOCK_MODE =", MOCK_MODE)

## 2. Charger les paires (original → anonymisé)  — **TODO**

Charge `audit_sample.csv` (brut) et ton `audit_sample_anonymized_<prenom>.csv`
(produit en async). Construis un DataFrame `pairs` avec 2 colonnes :
`original` et `anonymise` (colonne `manager_comments` de chaque fichier).

In [ ]:
DATA = Path("../data")

# TODO : charger les 2 CSV et construire `pairs` (colonnes: original, anonymise)
# pairs = pd.DataFrame({...})

# Astuce : ajoute 2-3 exemples FR piégés (noms FR NON masqués) pour vérifier
# que le juge rattrape ce que spaCy anglais a laissé passer.
raise NotImplementedError("Construis le DataFrame `pairs`")

## 3. Le prompt du juge — **TODO**

⚠️ **Piège à éviter.** Si tu demandes directement *« reste-t-il des PII dans ce
texte anonymisé ? »*, un petit modèle (`llama3.2:3b`) suppose qu'« anonymisé =
propre » et répond presque toujours **non** — il rate les noms FR. La parade qui
marche :

1. **Reformule en extraction** : demande-lui de *lister* les noms (prénom + nom),
   emails et téléphones encore **EN CLAIR** (pas un verdict oui/non).
2. **Donne un exemple guidé (few-shot)** : une paire (texte → bonne réponse JSON)
   que tu passeras dans `messages` **avant** le vrai texte.
3. **Précise** que les jetons déjà masqués `[PERSON_xxx]` et les villes/rôles ne
   comptent pas.

Définis `SYSTEME`, `SCHEMA` et l'exemple few-shot ci-dessous.

In [ ]:
# TODO : rédige SYSTEME (consigne d'EXTRACTION, pas de jugement) et SCHEMA.
# SYSTEME = "Tu es un extracteur d'entités. Liste les noms (prénom+nom), emails,
#            téléphones encore EN CLAIR. Ignore les jetons [PERSON_xxx], les
#            villes et les rôles. Donne aussi une lisibilité 1-5. JSON uniquement."
# SCHEMA  = '{"noms_en_clair": [...], "emails": [...], "telephones": [...], "lisibilite": 1-5}'

# TODO : un exemple few-shot (texte -> bonne réponse JSON) passé dans messages
# FEWSHOT_USER = "Texte: '...'"
# FEWSHOT_ASSISTANT = '{"noms_en_clair": [...], ...}'


# --- Fournis (plomberie robuste, ne pas réécrire) ---
_MASK_RE = re.compile(r"person_|email|phone|\[|\]", re.I)


def _strip_masks(items: list | None) -> list:
    """Retire les jetons déjà masqués que le LLM reliste parfois."""
    return [x for x in (items or []) if not _MASK_RE.search(str(x))]


def _parse_json(txt: str) -> dict:
    """Extrait le 1er objet JSON d'une réponse LLM (fourni — robuste)."""
    m = re.search(r"\{.*\}", txt, re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}

## 4. La fonction `judge_comment` — **TODO**

Deux chemins :
- **MOCK_MODE** : une heuristique simple (regex « Prénom Nom » hors crochets).
- **Ollama** : construis `messages = [SYSTEME, few-shot user, few-shot assistant,
  vrai texte]`, `requests.post(..., format="json", options={"temperature": 0})`,
  puis `_parse_json(...)`, **filtre avec `_strip_masks(...)`**, et reconstruis
  `pii_residuelle` / `types_pii` à partir des listes extraites.

> Le schéma de retour attendu par l'agrégation (cellule 5) reste :
> `{"pii_residuelle": bool, "types_pii": [...], "lisibilite": int}`.

In [ ]:
def judge_comment(original: str, anonymise: str) -> dict:
    """Juge un commentaire anonymisé : PII résiduelle + lisibilité.

    Retour attendu : {"pii_residuelle": bool, "types_pii": [...], "lisibilite": int}
    """
    if MOCK_MODE:
        # TODO : repli heuristique (regex "Prénom Nom" hors [crochets]),
        #        renvoyer un dict au schéma ci-dessus.
        raise NotImplementedError("Implémente le repli MOCK")
    # TODO :
    #  1) payload : messages = [SYSTEME, FEWSHOT_USER, FEWSHOT_ASSISTANT, texte]
    #  2) requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT_S) -> _parse_json
    #  3) noms/emails/tels = _strip_masks(raw.get("noms_en_clair")) etc.
    #  4) types_pii = ["nom"] si noms (+ "email", "telephone") ; pii_residuelle = bool(types_pii)
    raise NotImplementedError("Implémente l'appel Ollama")

## 5. Lancer sur un échantillon + agréger — **TODO**

In [ ]:
# TODO : appliquer judge_comment sur ~15-20 paires, agréger :
#  - nombre de PII résiduelles signalées
#  - lisibilité moyenne
#  - zoom sur les commentaires FR (le juge doit rattraper les noms FR)
raise NotImplementedError("Lance le juge et agrège les verdicts")

## ✅ Vérification (checklist)

- [ ] Le notebook tourne en `MOCK_MODE` **et** avec Ollama
- [ ] Le juge **rattrape** des noms FR ratés par `en_core_web_md`
- [ ] Je fige `temperature=0` (verdicts reproductibles)
- [ ] Je peux citer 2 raisons de **ne pas** déployer un anonymiseur 100 % LLM
- [ ] J'ai noté pourquoi un LLM **local** est préférable au cloud sur des PII

## ⭐ Pour aller plus loin
- Faire **voter** 2 modèles et ne garder les alertes que si concordance.
- Comparer le verdict du juge avec ce que `fr_core_news_md` aurait attrapé.